# Normalizacao

In [24]:
# pip install pandas numpy pyarrow duckdb

### Imports

In [25]:
import pandas as pd
import numpy as np
import gc
import duckdb

### Despesas

#### Função

In [18]:
def normalizar_empenhos(df: pd.DataFrame) -> dict:
    # -------------------------------------------------------------------------
    # 0. Limpeza de Nomes de Colunas (Remove caractere BOM \ufeff da coluna municipio)
    # -------------------------------------------------------------------------
    df.columns = df.columns.str.replace('\ufeff', '').str.strip()

    # -------------------------------------------------------------------------
    # 1FN: Ajustes de Tipagem e Conversão
    # -------------------------------------------------------------------------
    colunas_valores = ['valor_empenhado', 'valor_liquidado', 'valor_pago']
    for col in colunas_valores:
        if col in df.columns and (df[col].dtype == 'object' or df[col].dtype == 'string'):
            df[col] = (
                df[col]
                .astype(str)
                .str.replace('.', '', regex=False)
                .str.replace(',', '.', regex=False)
            )
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    if 'data_empenho' in df.columns:
        df['data_empenho'] = pd.to_datetime(df['data_empenho'], errors='coerce')

    # -------------------------------------------------------------------------
    # 2FN e 3FN: Extração das Tabelas de Dimensão
    # -------------------------------------------------------------------------
    dim_unidade_gestora = (
        df[['codigo_unidade_gestora', 'descricao_unidade_gestora', 'municipio']]
        .drop_duplicates(subset=['codigo_unidade_gestora'])
        .reset_index(drop=True)
    )

    dim_unidade_orcamentaria = (
        df[['codigo_unidade_orcamentaria', 'descricao_unidade_orcamentaria']]
        .drop_duplicates(subset=['codigo_unidade_orcamentaria'])
        .reset_index(drop=True)
    )

    dim_credor = (
        df[['cpf_cnpj', 'nome_credor']]
        .drop_duplicates(subset=['cpf_cnpj'])
        .reset_index(drop=True)
    )

    dim_funcao = (
        df[['codigo_funcao', 'funcao']]
        .drop_duplicates(subset=['codigo_funcao'])
        .reset_index(drop=True)
    )

    dim_subfuncao = (
        df[['codigo_subfuncao', 'subfuncao']]
        .drop_duplicates(subset=['codigo_subfuncao'])
        .reset_index(drop=True)
    )

    dim_programa = (
        df[['codigo_programa', 'programa']]
        .drop_duplicates(subset=['codigo_programa'])
        .reset_index(drop=True)
    )

    dim_acao = (
        df[['codigo_acao', 'acao']]
        .drop_duplicates(subset=['codigo_acao'])
        .reset_index(drop=True)
    )

    dim_categoria_economica = (
        df[['codigo_categoria_economica', 'categoria_economica']]
        .drop_duplicates(subset=['codigo_categoria_economica'])
        .reset_index(drop=True)
    )

    dim_natureza = (
        df[['codigo_natureza', 'grupo_natureza_despesa']]
        .drop_duplicates(subset=['codigo_natureza'])
        .reset_index(drop=True)
    )

    dim_modalidade_aplicacao = (
        df[['codigo_modalidade_aplicacao', 'modalidade_aplicacao']]
        .drop_duplicates(subset=['codigo_modalidade_aplicacao'])
        .reset_index(drop=True)
    )

    dim_elemento_despesa = (
        df[['codigo_elemento_despesa', 'elemento_despesa']]
        .drop_duplicates(subset=['codigo_elemento_despesa'])
        .reset_index(drop=True)
    )

    dim_fonte_recurso = (
        df[['codigo_fonte_recurso', 'descricao_fonte_recurso']]
        .drop_duplicates(subset=['codigo_fonte_recurso'])
        .reset_index(drop=True)
    )

    dim_co = (
        df[['co', 'descricao_co']]
        .dropna(subset=['co'])
        .drop_duplicates(subset=['co'])
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # Tabela Fato
    # -------------------------------------------------------------------------
    colunas_fato = [
        'numero_empenho', 'ano_referencia', 'data_empenho', 'mes',
        'codigo_unidade_gestora', 'codigo_unidade_orcamentaria', 'cpf_cnpj',
        'codigo_funcao', 'codigo_subfuncao', 'codigo_programa', 'codigo_acao',
        'codigo_categoria_economica', 'codigo_natureza', 'codigo_modalidade_aplicacao',
        'codigo_elemento_despesa', 'codigo_subelemento', 'codigo_subelemento_exibicao',
        'codigo_fonte_recurso', 'co', 'numero_licitacao', 'modalidade_licitacao',
        'numero_obra', 'valor_empenhado', 'valor_liquidado', 'valor_pago',
        'historico', 'ano_fonte', 'arquivo_origem'
    ]

    fato_empenhos = df[[col for col in colunas_fato if col in df.columns]].copy()

    return {
        'fato_empenhos': fato_empenhos,
        'dim_unidade_gestora': dim_unidade_gestora,
        'dim_unidade_orcamentaria': dim_unidade_orcamentaria,
        'dim_credor': dim_credor,
        'dim_funcao': dim_funcao,
        'dim_subfuncao': dim_subfuncao,
        'dim_programa': dim_programa,
        'dim_acao': dim_acao,
        'dim_categoria_economica': dim_categoria_economica,
        'dim_natureza': dim_natureza,
        'dim_modalidade_aplicacao': dim_modalidade_aplicacao,
        'dim_elemento_despesa': dim_elemento_despesa,
        'dim_fonte_recurso': dim_fonte_recurso,
        'dim_co': dim_co
    }

#### Carregando base de dados

In [19]:
df_despesas = pd.read_csv('dados_consolidados/despesas_2021_2025.csv')
df_despesas.head()

C:\Users\eu\AppData\Local\Temp\ipykernel_21616\1382293120.py:1: DtypeWarning: Columns (0: numero_licitacao, 1: descricao_co) have mixed types. Specify dtype option on import or set low_memory=False.
  df_despesas = pd.read_csv('dados_consolidados/despesas_2021_2025.csv')


,﻿municipio,codigo_unidade_gestora,descricao_unidade_gestora,numero_empenho,data_empenho,mes,cpf_cnpj,nome_credor,valor_empenhado,valor_liquidado,...,modalidade_licitacao,numero_obra,historico,codigo_fonte_recurso,descricao_fonte_recurso,ano_fonte,co,descricao_co,ano_referencia,arquivo_origem
0,Cabedelo,201040.0,Prefeitura Municipal de Cabedelo,1,2021-01-07,01-Janeiro,293946,BANCO DO BRASIL S/A,15.000,"14.940,75",...,Sem Licitação,NaN,Valor que se empenha para fazer face a despesa...,1001,Recursos Ordinários - Recursos do Exercício Co...,NaN,NaN,NaN,2021,despesas-2021.csv
1,Cabedelo,201040.0,Prefeitura Municipal de Cabedelo,2,2021-01-07,01-Janeiro,360305003987,CAIXA ECONOMICA FEDERAL,15.000,"14.995,25",...,Sem Licitação,NaN,Valor que se empenha para fazer face a despesa...,1001,Recursos Ordinários - Recursos do Exercício Co...,NaN,NaN,NaN,2021,despesas-2021.csv
2,Cabedelo,201040.0,Prefeitura Municipal de Cabedelo,3,2021-01-07,01-Janeiro,293946,BANCO DO BRASIL S/A,5.000,"4.994,5",...,Sem Licitação,NaN,Valor que se empenha para fazer face a despesa...,1001,Recursos Ordinários - Recursos do Exercício Co...,NaN,NaN,NaN,2021,despesas-2021.csv
3,Cabedelo,201040.0,Prefeitura Municipal de Cabedelo,14,2021-01-07,01-Janeiro,293946,BANCO DO BRASIL S/A,1.000,"41,6",...,Sem Licitação,NaN,Valor que se empenha para fazer face a despesa...,1001,Recursos Ordinários - Recursos do Exercício Co...,NaN,NaN,NaN,2021,despesas-2021.csv
4,Cabedelo,201040.0,Prefeitura Municipal de Cabedelo,15,2021-01-07,01-Janeiro,394460040950,MIN.DA FAZENDA/SEC. DO TESOURO NACIONAL,200.000,"195.259,08",...,Sem Licitação,NaN,Valor que se empenha para fazer face a despesa...,1001,Recursos Ordinários - Recursos do Exercício Co...,NaN,NaN,NaN,2021,despesas-2021.csv


#### Normalizando

In [21]:
modelos = normalizar_empenhos(df_despesas)

In [13]:
del df_despesas
gc.collect()

9533

In [28]:
for nome_tabela, df_tabela in modelos.items():
    print(f"Salvando {nome_tabela} ({len(df_tabela):,} linhas)...")
    
    # O DuckDB reconhece a variável 'df_tabela' do Python e exporta direto para Parquet
    duckdb.sql(f"COPY df_tabela TO '{nome_tabela}.parquet' (FORMAT PARQUET)")

print("Todas as tabelas foram salvas em Parquet com sucesso!")

Salvando fato_empenhos (10,753,139 linhas)...
Salvando dim_unidade_gestora (646 linhas)...
Salvando dim_unidade_orcamentaria (624 linhas)...
Salvando dim_credor (571,362 linhas)...
Salvando dim_funcao (27 linhas)...
Salvando dim_subfuncao (98 linhas)...
Salvando dim_programa (712 linhas)...
Salvando dim_acao (1,858 linhas)...
Salvando dim_categoria_economica (3 linhas)...
Salvando dim_natureza (7 linhas)...
Salvando dim_modalidade_aplicacao (18 linhas)...
Salvando dim_elemento_despesa (57 linhas)...
Salvando dim_fonte_recurso (135 linhas)...
Salvando dim_co (17 linhas)...
Todas as tabelas foram salvas em Parquet com sucesso!
